# Cosmos property profiles to Silver

Flattens the mirrored property documents into one property-profile table and explodes `inspections[]` into a separate inspection fact. If your mirror exposes a schema folder, include it in `source_table_path`.

In [ ]:
%%configure -f
{
  "defaultLakehouse": { "name": "SilverLakehouse" }
}

In [ ]:
cosmos_mirror_item = "Property Profiles Mirror"
source_table_path = "Tables/properties"

In [ ]:
import requests
from pyspark.sql import functions as F

import notebookutils
workspace_id = notebookutils.runtime.context["currentWorkspaceId"]

def resolve_item_id(display_name, item_type):
    token = notebookutils.credentials.getToken("pbi")
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items?type={item_type}"
    response = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=60)
    response.raise_for_status()
    matches = [item for item in response.json()["value"] if item["displayName"] == display_name]
    if len(matches) != 1:
        raise ValueError(f"Expected one {item_type} named '{display_name}', found {len(matches)}")
    return matches[0]["id"]

mirror_id = resolve_item_id(cosmos_mirror_item, "MirroredDatabase")
source_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{mirror_id}/{source_table_path}"
raw = spark.read.format("delta").load(source_path)
raw.printSchema()
print(f"Raw profiles: {raw.count()}")

In [ ]:
profiles = raw.select(
    F.col("parcelId").alias("parcel_id"),
    F.col("documentType").alias("document_type"),
    F.col("synthetic").cast("boolean").alias("synthetic"),
    F.col("neighborhoodId").alias("neighborhood_id"),
    F.col("neighborhoodName").alias("neighborhood_name"),
    F.col("jurisdiction.countryCode").alias("country_code"),
    F.col("jurisdiction.regionCode").alias("region_code"),
    F.col("jurisdiction.currencyCode").alias("currency_code"),
    F.col("location.coordinates").getItem(0).cast("double").alias("longitude"),
    F.col("location.coordinates").getItem(1).cast("double").alias("latitude"),
    F.col("site.syntheticAddress").alias("synthetic_address"),
    F.col("site.propertyClassCode").alias("property_class_code"),
    F.col("site.zoningCode").alias("zoning_code"),
    F.col("site.lotAreaSquareMetres").cast("double").alias("lot_area_square_metres"),
    F.col("building.type").alias("building_type"),
    F.col("building.yearBuilt").cast("int").alias("year_built"),
    F.col("building.floorAreaSquareMetres").cast("double").alias("floor_area_square_metres"),
    F.col("building.condition").alias("building_condition"),
    F.col("currentAssessment.taxYear").cast("int").alias("current_tax_year"),
    F.col("currentAssessment.assessedValue").cast("double").alias("current_assessed_value"),
    F.col("currentAssessment.confidenceScore").cast("double").alias("confidence_score"),
).dropna(subset=["parcel_id", "neighborhood_id"]).dropDuplicates(["parcel_id"])

(profiles.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("dbo.property_profile"))

In [ ]:
inspections = (raw
    .select(F.col("parcelId").alias("parcel_id"), F.explode_outer("inspections").alias("inspection"))
    .select(
        "parcel_id",
        F.col("inspection.inspectionId").alias("inspection_id"),
        F.to_date("inspection.inspectionDate").alias("inspection_date"),
        F.col("inspection.inspectionType").alias("inspection_type"),
        F.col("inspection.condition").alias("condition"),
        F.col("inspection.observations").alias("observations"),
        F.col("inspection.followUpRequired").cast("boolean").alias("follow_up_required"),
    )
    .dropna(subset=["parcel_id", "inspection_id"])
    .dropDuplicates(["inspection_id"]))

(inspections.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("dbo.fact_inspection"))

assert profiles.filter(F.col("synthetic") != True).count() == 0
assert profiles.filter(F.col("latitude").isNull() | F.col("longitude").isNull()).count() == 0
print(f"Profiles: {profiles.count()}, inspections: {inspections.count()}")